# Arquitectura y concurrencia · notebook de las clases 11.2 y 11.3

Este notebook acompaña las páginas 9 a 12 de la unidad 11. Es el lugar
**verboso**: cada tema se explica antes de la celda, la celda imprime
mensajes claros y, después, una celda «Salida esperada» dice qué debe
aparecer. Las páginas traen sólo las celdas cortas; aquí se repite el
contexto para que puedas ejecutar cada tema sin volver a leer la página.

**Reglas de las celdas.**

- Se ejecutan **en orden**, de arriba abajo. Cada celda de código empieza con
  un rótulo `# celda N.M`; ese rótulo es el que citan las páginas.
- Usan `await` de nivel superior: el kernel de Jupyter ya tiene un event loop
  corriendo, así que `asyncio.run(...)` truena aquí y nunca aparece.
- Los valores de «Salida esperada» son **cualitativos o rangos**. Los PID, los
  nombres de hilo y los milisegundos exactos cambian en cada máquina y en
  cada corrida; lo que debe repetirse es la **forma** del resultado.
- Las celdas 11.1 y 12.1 lanzan su propio servidor (`app_arq.py`, un solo
  worker) en un subproceso y lo cierran en un `finally`. No necesitas abrir
  otra terminal.

## 0.1 · Arranque: un lanzador reutilizable para el servidor de la clase

El servidor de las demostraciones D1 y D2 es `app_arq.py`, una aplicación
FastAPI que vive junto a este notebook. Guarda su estado **en la memoria del
proceso** (el evento de D2, los contadores), por eso se lanza con **un solo
worker** y sin `--reload`: con dos workers cada uno tendría su propia
memoria y las mediciones no significarían lo que dicen.

La celda de abajo sólo **define** dos funciones; no lanza nada todavía:

- `lanza_servidor(puerto=None, plazo=20.0)` ejecuta
  `sys.executable -m uvicorn app_arq:app --port PUERTO --workers 1` en un
  subproceso (el `--workers 1` explícito evita que una variable de entorno
  como `WEB_CONCURRENCY` active varios y parta el estado en memoria),
  con la carpeta de este notebook como directorio de trabajo. Si no das
  puerto, pide uno libre al sistema operativo. Espera, con un plazo, a que
  `/quien` responda. Falla con un mensaje claro si el puerto ya está ocupado
  o si el proceso termina antes de responder (y muestra el final de su log).
  Devuelve el proceso, con la dirección en `proc.url`.
- `detiene_servidor(proc)` pide el cierre con `terminate`, espera hasta 5 s y,
  sólo si no cerró, lo mata con `kill`. Es seguro llamarla dos veces.

El patrón de uso, en todas las celdas que miden, es siempre el mismo:

```text
proc = lanza_servidor()
try:
    ...   # la medición
finally:
    detiene_servidor(proc)
```

Así el servidor se cierra aunque la medición falle o interrumpas el kernel.

In [ ]:
# celda 0.1
import os
import socket
import subprocess
import sys
import tempfile
import time
import urllib.request
from pathlib import Path

CARPETA = Path.cwd()   # Jupyter arranca el kernel en la carpeta del notebook


def puerto_libre():
    """Un puerto TCP que el sistema operativo declara libre ahora mismo."""
    with socket.socket() as s:
        s.bind(("127.0.0.1", 0))
        return s.getsockname()[1]


def _ocupado(puerto):
    with socket.socket() as s:
        s.settimeout(0.5)
        return s.connect_ex(("127.0.0.1", puerto)) == 0


def _final_del_log(proc, lineas=15):
    try:
        proc.log.flush()
        texto = Path(proc.log.name).read_text(errors="replace")
    except Exception:
        return "(sin log)"
    return "\n".join(texto.splitlines()[-lineas:]) or "(log vacío)"


def lanza_servidor(puerto=None, plazo=20.0):
    """Arranca app_arq con UN worker y espera a que /quien responda."""
    if not (CARPETA / "app_arq.py").is_file():
        raise FileNotFoundError(f"no encuentro app_arq.py en {CARPETA}")
    puerto = puerto or puerto_libre()
    if _ocupado(puerto):
        raise RuntimeError(f"el puerto {puerto} ya está ocupado: elige otro")
    log = tempfile.NamedTemporaryFile("w+", prefix="app_arq_", suffix=".log",
                                      delete=False)
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "app_arq:app",
         "--port", str(puerto), "--workers", "1",
         "--log-level", "warning"],
        cwd=CARPETA, stdout=log, stderr=subprocess.STDOUT)
    proc.log = log
    proc.url = f"http://127.0.0.1:{puerto}"
    try:
        limite = time.monotonic() + plazo
        while time.monotonic() < limite:
            if proc.poll() is not None:
                raise RuntimeError(f"el servidor terminó antes de responder "
                                   f"(código {proc.returncode}). Final de su "
                                   f"log:\n{_final_del_log(proc)}")
            try:
                with urllib.request.urlopen(proc.url + "/quien", timeout=1) as r:
                    if r.status == 200:
                        return proc
            except OSError:
                time.sleep(0.1)
        raise TimeoutError(f"el servidor no respondió /quien en {plazo} s")
    except BaseException:       # también Ctrl-C / interrupción del kernel
        detiene_servidor(proc)  # cierra el hijo y borra el log
        raise


def detiene_servidor(proc):
    """terminate, espera acotada y, si hace falta, kill. Idempotente."""
    if proc is None:
        return
    if proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()
            proc.wait(timeout=5)
    log = getattr(proc, "log", None)
    if log is not None and not log.closed:
        log.close()
        try:
            os.unlink(log.name)
        except OSError:
            pass


print("lanza_servidor y detiene_servidor definidas")
print("carpeta de trabajo:", CARPETA)
print("python del kernel :", sys.executable)

### Salida esperada de la celda 0.1

Tres renglones: el aviso de que las dos funciones quedaron definidas, la
carpeta donde está este notebook (debe contener `app_arq.py`) y la ruta del
Python del kernel (debe ser el de tu ambiente `uv`, no el del sistema). Si
la carpeta no es la del notebook, abre el notebook desde su carpeta o cambia
el directorio del kernel antes de seguir.

## 9.1 · Relojes locales: restar marcas no mide la latencia

**Qué se va a ver.** Dos componentes, A y B, tienen cada uno su propio reloj
local. El reloj de B marca **40 ms menos** que el tiempo real (está atrasado).
A envía un mensaje a B; el mensaje tarda **20 ms** en llegar (valor supuesto
del ejemplo). Cada componente anota la hora de **su** reloj: A al enviar, B
al recibir.

Es una simulación con enteros en milisegundos, no una medición: no usa el
reloj real de tu máquina. Lo que importa es la cuenta, y es exacta:

- envío en A: tiempo real 100 ms, marca de A = 100
- llegada a B: tiempo real 120 ms, marca de B = 120 − 40 = 80
- resta de marcas: 80 − 100 = **−20 ms**

La resta es negativa: «el mensaje llegó antes de enviarse». No es un error
del programa; es que la resta mezcla dos relojes. Vale
`latencia + desfase = 20 + (−40) = −20`. Sin conocer el desfase (y su
incertidumbre) la resta no dice cuánto tardó el mensaje.

In [ ]:
# celda 9.1
LATENCIA_MS = 20          # supuesto: lo que tarda el mensaje en tiempo real
DESFASE_B_MS = -40        # supuesto: el reloj de B marca 40 ms menos que el real


class Componente:
    """Un componente con reloj local = tiempo real + desfase."""

    def __init__(self, nombre, desfase_ms):
        self.nombre = nombre
        self.desfase_ms = desfase_ms

    def marca(self, real_ms):
        return real_ms + self.desfase_ms


a = Componente("A", 0)
b = Componente("B", DESFASE_B_MS)

envio_real = 100                       # A envía en el instante real 100 ms
llegada_real = envio_real + LATENCIA_MS

marca_a = a.marca(envio_real)          # lo que A anota
marca_b = b.marca(llegada_real)        # lo que B anota

print(f"A anota al enviar : {marca_a} ms  (reloj de A)")
print(f"B anota al recibir: {marca_b} ms  (reloj de B)")
print(f"resta de marcas   : {marca_b} - {marca_a} = {marca_b - marca_a} ms")
print(f"latencia real     : {llegada_real - envio_real} ms (la conocemos sólo porque simulamos)")
print(f"latencia + desfase: {LATENCIA_MS} + ({DESFASE_B_MS}) = {LATENCIA_MS + DESFASE_B_MS} ms")
assert (marca_a, marca_b, marca_b - marca_a) == (100, 80, -20)

### Salida esperada de la celda 9.1

Los valores son exactos porque es una simulación con enteros:

- A anota 100 ms y B anota 80 ms.
- La resta de marcas da **−20 ms** (negativa).
- La latencia real es 20 ms y `latencia + desfase` también da −20 ms.

**Lectura.** La diferencia entre marcas de dos máquinas no es la latencia. El
orden de eventos entre máquinas se reconstruye con **mensajes** («A envía,
luego B recibe»), no comparando relojes. Si al ejecutar la celda no ves el
−20, revisa que no hayas cambiado `DESFASE_B_MS` o `LATENCIA_MS`.

## 10.1 · Quién ejecuta: 4 hilos frente a 2 procesos

**Qué se va a ver.** El mismo trabajo, **4 tareas** de cálculo puro de
Python (cada una del orden de 100 ms, según tu CPU), se manda dos veces:

1. a un `ThreadPoolExecutor` de **4 hilos** (un solo proceso);
2. a un `ProcessPoolExecutor` de **2 procesos**, con un contexto de arranque
   explícito (`spawn`) para que se comporte igual en cualquier sistema.

Cada tarea devuelve **quién la ejecutó**: el PID del proceso y el nombre del
hilo. La función vive en `trabajo_arq.py` (junto a este notebook) porque el
proceso hijo la busca **importando** ese módulo; una función definida en una
celda no es importable por un hijo `spawn`.

Lo que la celda muestra es la **pertenencia**: con hilos, un PID y varios
nombres de hilo; con procesos, varios PID. La celda también imprime el
número de CPU lógicas de tu máquina, que es el tope de ejecución simultánea
del hardware. **No promete aceleración**: en CPython con GIL, 4 hilos de
cálculo puro no corren bytecode a la vez, y 2 procesos pagan el costo de
arrancar. La celda no compara tiempos.

In [ ]:
# celda 10.1
import multiprocessing
import os
from concurrent.futures import ProcessPoolExecutor, ThreadPoolExecutor

import threading

from trabajo_arq import tarea

N_TAREAS = 4
cpus = getattr(os, "process_cpu_count", os.cpu_count)()
print(f"CPU lógicas disponibles para este proceso: {cpus}")

# 1. cuatro hilos, un proceso. La barrera hace que ninguna tarea empiece su
# cálculo hasta que las 4 estén dentro de un hilo distinto (si no, un hilo
# libre podría tomar dos tareas y verías 3 nombres de hilo)
barrera = threading.Barrier(N_TAREAS, timeout=30)


def tarea_con_barrera(i):
    barrera.wait()
    return tarea(i)


with ThreadPoolExecutor(max_workers=4, thread_name_prefix="hilo") as pool:
    con_hilos = list(pool.map(tarea_con_barrera, range(N_TAREAS)))

print("\n4 hilos:")
for r in con_hilos:
    print(f"  tarea {r['tarea']}: PID {r['pid']}  hilo {r['hilo']}")
pids_h = {r["pid"] for r in con_hilos}
hilos_h = {r["hilo"] for r in con_hilos}
print(f"  -> {len(pids_h)} PID y {len(hilos_h)} nombres de hilo distintos")

# 2. dos procesos (contexto explícito)
ctx = multiprocessing.get_context("spawn")
with ProcessPoolExecutor(max_workers=2, mp_context=ctx) as pool:
    con_procesos = list(pool.map(tarea, range(N_TAREAS)))

print("\n2 procesos:")
for r in con_procesos:
    print(f"  tarea {r['tarea']}: PID {r['pid']}  hilo {r['hilo']}")
pids_p = {r["pid"] for r in con_procesos}
print(f"  -> {len(pids_p)} PID distintos (y ninguno es el del kernel: {os.getpid()})")

assert len(pids_h) == 1 and len(hilos_h) == 4
assert 1 <= len(pids_p) <= 2 and os.getpid() not in pids_p

### Salida esperada de la celda 10.1

- Primero el número de CPU lógicas (un entero; en la máquina del profesor, 8).
- **4 hilos:** las cuatro tareas muestran **el mismo PID** (el del kernel) y
  cuatro nombres de hilo distintos (`hilo_0` a `hilo_3`). Cierra con
  «1 PID y 4 nombres de hilo distintos».
- **2 procesos:** las cuatro tareas muestran **PID distintos del kernel**:
  normalmente dos PID, cada uno repetido (cada proceso atendió dos tareas), y
  el nombre de hilo es `MainThread` en ambos (cada proceso hijo ejecuta en su
  hilo principal). Cierra con «2 PID distintos».

Los valores concretos de PID cambian en cada ejecución. Si ves un solo PID
en la segunda parte, un proceso atendió las cuatro tareas (es posible, no es
un error); repite la celda. **Lectura.** Hilos y procesos responden a
«quién ejecuta»; cuántos de ellos corren a la vez lo deciden además el
planificador del sistema operativo, las CPU y el GIL, no el tamaño del pool.

## 11.1 · D1: `async def` frente a `def` con 12 peticiones simultáneas

**Qué se va a ver.** El servidor tiene dos endpoints que hacen lo mismo,
dormir `s` segundos con `time.sleep` (una espera que **bloquea** el hilo que
la ejecuta), y se diferencian sólo en cómo están declarados:

- `/async-bloqueante` es `async def`: FastAPI lo ejecuta **en el hilo del
  event loop**. Mientras duerme, el loop no atiende nada más.
- `/def-bloqueante` es `def`: FastAPI lo **desplaza** a un pool de hilos (el
  limitador de AnyIO admite 40 a la vez por defecto), y el loop queda libre.

La celda manda **12 peticiones simultáneas** con `w = 0.5 s` a cada endpoint,
una modalidad tras otra, contra un **solo worker**, y mide con
`time.monotonic` el **total**: desde que salen las 12 hasta que todas
contestan.

**El latido, paso a paso.** Para comprobar que el loop quedó bloqueado, el
servidor agrega al loop una tarea más, el latido:

1. Hace `await asyncio.sleep(0.01)` y, al despertar, anota cuánto tiempo pasó
   desde la vez anterior. Ese tiempo es el **hueco**.
2. Con el loop libre, el latido recibe turno en cuanto vence su espera de
   10 ms: el hueco es de unos 10 ms.
3. Con `async def`, cada petición hace `time.sleep(0.5)` sin `await`: el hilo
   del loop duerme y no cede el turno. El latido ya cumplió sus 10 ms, así
   que está **lista** (como en los Gantt: puede avanzar y no tiene turno).
4. El loop pasa de una petición a la siguiente sin dejar turno al latido, que
   no corre hasta que terminan las 12: hueco aproximado de 12·0.5 = 6 s.
5. Con `def`, el `sleep` ocurre en hilos aparte, el loop sigue libre y el
   latido corre cada 10 ms.

`GET /latido` devuelve el **hueco máximo** (`max_gap`) y el número de ticks;
`POST /reinicia` los pone a cero. La celda reinicia antes de cada modalidad y
lee `/latido` al terminar.

Cuenta aproximada: con `async def` las esperas se ejecutan una tras otra,
`T ≈ 12·w = 6 s`. Además el loop corre los 12 handlers **seguidos**, sin
dejar correr a la tarea del latido entre uno y otro, así que el hueco máximo
es de **≈ 12·w ≈ 6 s**, no de `w`; con `def` corren a la vez en hilos, `T ≈ w = 0.5 s` (más
algo de costo fijo) y el hueco sigue siendo pequeño (< 0.1 s). Son
aproximaciones; lo que se verifica son rangos. Antes de medir, la celda abre
las 12 conexiones con una petición a `/quien` cada una (calentamiento), para
que el tiempo de establecerlas no se mezcle con la espera. El servidor se
lanza dentro de la celda y se cierra en un `finally`.

In [ ]:
# celda 11.1
import asyncio
import httpx

W = 0.5          # segundos que duerme cada petición
N_PETICIONES = 12

LIMITES = httpx.Limits(max_connections=20, max_keepalive_connections=20)
PLAZOS = httpx.Timeout(30.0)       # lectura de 30 s: mayor que 2*12*w = 12 s


async def mide_modalidad(base, ruta):
    """12 peticiones simultáneas a `ruta`; devuelve (total, hueco máximo del latido)."""
    async with httpx.AsyncClient(limits=LIMITES, timeout=PLAZOS) as cliente:
        # calienta las 12 conexiones (keep-alive) antes de medir
        await asyncio.gather(*(cliente.get(base + "/quien") for _ in range(N_PETICIONES)))
        await cliente.post(base + "/reinicia")          # latido a cero

        t0 = time.monotonic()
        respuestas = await asyncio.gather(*(
            cliente.get(f"{base}{ruta}?s={W}") for _ in range(N_PETICIONES)))
        total = time.monotonic() - t0

        latido = (await cliente.get(base + "/latido")).json()
    assert all(r.status_code == 200 for r in respuestas)
    return total, latido["max_gap"], latido["ticks"]


proc = lanza_servidor()
try:
    print(f"servidor en {proc.url} (un worker)")
    resultados = {}
    async with asyncio.timeout(60):                      # plazo global
        for etiqueta, ruta in (("async def", "/async-bloqueante"),
                               ("def", "/def-bloqueante")):
            print(f"midiendo {etiqueta} ...")
            resultados[etiqueta] = await mide_modalidad(proc.url, ruta)
finally:
    detiene_servidor(proc)

print(f"\n{N_PETICIONES} peticiones simultáneas, w = {W} s")
print(f"{'modalidad':<10} {'total (s)':>10} {'hueco máx. del latido (s)':>26}")
for etiqueta, (total, hueco, ticks) in resultados.items():
    print(f"{etiqueta:<10} {total:>10.2f} {hueco:>26.3f}")
print(f"esperado: async def ~ {N_PETICIONES * W:.1f} s y hueco ~ {N_PETICIONES * W:.1f} s; "
      f"def ~ {W:.1f} s y hueco < 0.1 s")

t_async, h_async, _ = resultados["async def"]
t_def, h_def, _ = resultados["def"]
assert t_async >= 8 * W and t_def <= 3 * W, "los totales salen del rango esperado"
assert h_async >= 4 * W and h_def < 0.1, "el hueco del latido sale del rango esperado"

### Salida esperada de la celda 11.1

Una tabla con dos renglones y valores **en rangos**, no exactos:

- `async def`: total **cercano a 6 s** (al menos 4 s; las 12 esperas se
  sirvieron una tras otra) y hueco máximo del latido **también cercano a
  6 s** (al menos 2 s): el loop ejecutó los 12 handlers seguidos y la tarea
  del latido no pudo volver a correr en todo ese tiempo (no sólo durante una
  espera de `w = 0.5 s`).
- `def`: total **cercano a 0.5 s** (no más de 1.5 s) y hueco máximo
  **menor que 0.1 s**: las esperas corrieron en hilos y el loop siguió
  corriendo su latido cada ~10 ms.

Si `async def` sale cerca de 0.5 s o `def` cerca de 6 s, revisa que el
servidor tenga un solo worker y que no cambiaste `time.sleep` por
`await asyncio.sleep` (esa versión **no** bloquea el loop). **Lectura.** No
se bloqueó la CPU, se bloqueó el **loop**: mientras un `async def` llama a
`time.sleep`, ninguna otra corrutina avanza, incluida la del latido. Un `def`
llamado desde dentro de un `async def` no se desplaza: corre en el hilo del
loop.

## 11.2 · Presupuesto de conexiones: R × N × k

**Qué se va a ver.** Una cuenta, sin servidor. Un despliegue con
**R = 3** réplicas, cada una con **N = 4** workers, y cada worker con un
pool local de hasta **k = 10** conexiones a la base de datos. Cada worker crea
**su propio** pool (no se supone que un pool se comparta entre procesos), de
modo que el máximo configurado contra la base es el producto
`R × N × k = 3 × 4 × 10 = 120`.

Esa cifra se compara con lo que admite una base PostgreSQL con el valor
habitual `max_connections = 100`: con 3 plazas reservadas para superusuario
quedan **97** para el resto (comprueba los tuyos con `SHOW max_connections;`
y `SHOW superuser_reserved_connections;`). La cuenta supone que 10 es el
máximo **total** por pool y que todas las conexiones van contra el mismo
servidor.

In [ ]:
# celda 11.2
R, N, k = 3, 4, 10                    # réplicas, workers por réplica, conexiones por pool
max_connections = 100                 # valor habitual de PostgreSQL
reservadas = 3                        # plazas de superusuario

potencial = R * N * k
disponibles = max_connections - reservadas

print(f"capacidad potencial: R x N x k = {R} x {N} x {k} = {potencial} conexiones")
print(f"disponibles en la base: {max_connections} - {reservadas} = {disponibles}")
print(f"el máximo configurado excede lo disponible en {potencial - disponibles}")
print("¿cabe todo el máximo a la vez?", potencial <= disponibles)

### Salida esperada de la celda 11.2

Con los valores de partida (si los cambias, cambian las cifras):

- `120` conexiones de capacidad potencial.
- `97` disponibles.
- El máximo configurado excede lo disponible en **23**, y la respuesta a
  «¿cabe todo el máximo a la vez?» es `False`.

**Lectura.** 120 es capacidad **potencial**, no uso: mientras los pools no
se llenen, todo funciona. El problema aparece bajo carga, cuando varios
pools piden su máximo a la vez, y escalar réplicas multiplica esa presión
sobre la base. Cambia `R`, `N` o `k` y repite para ver cuándo la cuenta cabe
en 97: la celda no tiene aserciones, así que cualquier valor se imprime.

## 12.1 · D2: cómo llegan los resultados (polling, long polling y SSE)

**Qué se va a ver.** Un evento que ocurre en el servidor **10 s** después de
`POST /inicia?t=10`. Tres maneras de enterarse, con **20 clientes** en cada
una, cada modalidad **por separado** (reinicia el servidor y vuelve a
iniciar el evento):

- **polling**: cada cliente pregunta a `/listo` cada **Δ = 3 s**, con
  reloj monotónico, y no hace consulta inicial: consulta en 3, 6, 9 y 12 s.
  Cuenta esperada por cliente: 4 consultas (3 sin novedades) y una demora de
  `Δ·⌈T/Δ⌉ − T = 2 s`; con 20 clientes, 80 consultas y 60 sin novedades;
- **long polling**: una sola petición a `/espera` que el servidor mantiene
  abierta hasta el evento; demora esperada: casi cero;
- **SSE**: una respuesta HTTP continua en `/eventos` (streaming) que envía
  latidos y el evento `listo`; demora esperada: casi cero.

Se cuenta, por modalidad: las **consultas** que recibió el servidor, la
**demora de detección** (desde el instante del evento hasta que el cliente se
entera, promedio de los 20), los **long polls pendientes** (sólo cuenta las peticiones a `/espera`) y las
**sesiones SSE abiertas** que reporta `/estado` a mitad de la espera, y los **sockets**
que conserva el cliente (el polling con keep-alive también conserva
conexiones, aunque no tenga peticiones pendientes). Son cosas distintas.

Cada modalidad dura ~10-12 s, así que la celda tarda cerca de 35 s. Todo el
bloque tiene un plazo global y el servidor se cierra en un `finally`.

In [ ]:
# celda 12.1
import asyncio
import httpx

CLIENTES = 20
T_EVENTO = 10.0     # segundos hasta el evento
DELTA = 3.0         # periodo del polling
PLAZOS = httpx.Timeout(30.0)
LIMITES = httpx.Limits(max_connections=CLIENTES + 5,
                       max_keepalive_connections=CLIENTES + 5)


def sockets_de(cliente):
    """Conexiones que el cliente httpx tiene abiertas; None si no se puede inspeccionar."""
    try:
        return len(cliente._transport._pool.connections)
    except AttributeError:
        return None


async def polling(base, t_evento, cliente):
    inicio = t_evento - T_EVENTO
    consultas, k = 0, 1
    while True:
        await asyncio.sleep(max(0.0, inicio + k * DELTA - time.monotonic()))
        r = await cliente.get(base + "/listo")
        consultas += 1
        if r.json()["listo"]:
            return time.monotonic() - t_evento, consultas
        k += 1


async def long_polling(base, t_evento, cliente):
    r = await cliente.get(base + "/espera")
    assert r.json()["listo"]
    return time.monotonic() - t_evento, 1


async def sse(base, t_evento, cliente):
    async with cliente.stream("GET", base + "/eventos") as r:
        async for linea in r.aiter_lines():
            if linea.startswith("event: listo"):
                return time.monotonic() - t_evento, 1
    raise RuntimeError("el flujo SSE terminó sin el evento listo")


async def corre_modalidad(base, funcion):
    """Una modalidad completa: reinicia, inicia el evento, 20 clientes, estado."""
    async with httpx.AsyncClient(timeout=PLAZOS) as control:
        await control.post(base + "/reinicia")
        # /inicia devuelve el instante monotónico en que el SERVIDOR fijó el
        # evento; la demora se mide contra ese instante (mismo reloj: el
        # cliente y el servidor corren en esta máquina)
        t_evento = (await control.post(base + f"/inicia?t={T_EVENTO}")).json()["evento"]
        clientes = [httpx.AsyncClient(limits=LIMITES, timeout=PLAZOS)
                    for _ in range(CLIENTES)]
        try:
            tareas = [asyncio.create_task(funcion(base, t_evento, c)) for c in clientes]
            await asyncio.sleep(T_EVENTO / 2)            # a mitad de la espera
            medio = (await control.get(base + "/estado")).json()
            cuentas = [sockets_de(c) for c in clientes]
            sockets = None if None in cuentas else sum(cuentas)
            resultados = await asyncio.gather(*tareas)
            await asyncio.sleep(0.5)                     # deja cerrar los flujos
            final = (await control.get(base + "/estado")).json()
        finally:
            for t in locals().get("tareas", []):
                t.cancel()
            for c in clientes:
                await c.aclose()
    demoras = [d for d, _ in resultados]
    return {"consultas": final["consultas"] if funcion is polling
            else sum(n for _, n in resultados),
            "demora": sum(demoras) / len(demoras),
            "pendientes": medio["esperando"], "sesiones_sse": medio["sse"],
            "sockets": sockets, "sse_al_final": final["sse"]}


MODALIDADES = (("polling", polling), ("long polling", long_polling), ("SSE", sse))

proc = lanza_servidor()
try:
    print(f"servidor en {proc.url} (un worker)")
    medidas = {}
    async with asyncio.timeout(60):                      # plazo global
        for nombre, funcion in MODALIDADES:
            print(f"corriendo {nombre} con {CLIENTES} clientes (~{T_EVENTO:.0f} s) ...")
            medidas[nombre] = await corre_modalidad(proc.url, funcion)
finally:
    detiene_servidor(proc)

print(f"\nevento en T = {T_EVENTO:.0f} s, {CLIENTES} clientes por modalidad")
print(f"{'modalidad':<13} {'consultas':>9} {'demora (s)':>10} "
      f"{'long polls pendientes':>21} {'sesiones SSE':>12} {'sockets cliente':>15}")
for nombre, m in medidas.items():
    sockets_txt = "no disponible" if m["sockets"] is None else m["sockets"]
    print(f"{nombre:<13} {m['consultas']:>9} {m['demora']:>10.2f} {m['pendientes']:>21} "
          f"{m['sesiones_sse']:>12} {sockets_txt:>15}")

assert 1.5 <= medidas["polling"]["demora"] <= 3.0
assert medidas["long polling"]["demora"] < 0.5 and medidas["SSE"]["demora"] < 0.5
assert medidas["SSE"]["sse_al_final"] == 0, "quedaron sesiones SSE abiertas"

### Salida esperada de la celda 12.1

Una tabla con tres renglones. Los rangos son los que verifica la celda:

- **polling:** `consultas` = **80** (20 clientes × 4; 60 sin novedades), demora
  **entre 1.5 y 3.0 s** (esperado ~2 s), **0 long polls pendientes**, 0 sesiones SSE y
  **~20 sockets** del cliente (conexiones keep-alive abiertas entre
  consultas aunque no haya nada pendiente).
- **long polling:** `consultas` = **20** (una petición por cliente), demora
  **menos de 0.5 s**, **20 long polls pendientes** a mitad de la espera (el servidor
  retiene esas peticiones abiertas), 0 sesiones SSE.
- **SSE:** `consultas` = **20** (una apertura por cliente), demora **menos de
  0.5 s**, 0 long polls pendientes, **20 sesiones SSE** abiertas a mitad de la espera y 20 sockets; al
  final no queda ninguna sesión.

Si el polling sale con 79 u 81 consultas, un cliente cayó justo en el borde
del evento; repite la celda. Si `sockets cliente` dice «no disponible», tu
versión de httpx guarda el pool en otro lugar: no afecta el resto de la
medición. **Lectura.** Cambian tres cantidades distintas: cuántas peticiones
hace el cliente, cuánto tarda en enterarse y qué retiene el servidor
mientras espera. El mecanismo no decide por sí solo si se retiene un hilo o
una tarea.